<h1 style="text-align: center;">TÉCNICO EM CIÊNCIA DE DADOS</h1>
<h1 style="text-align: center;">Filtragem e agrupamento de dados no PySpark</h1>
<br>

**Componente:** Fundamentos de Ambientes e Arquitetura de Dados
<br>
**Unidade Curricular:** Introdução ao PySpark e Processamento Distribuído
<br>
**Tema da Semana:** Filtragem e Agrupamento
<br>
**Semana 19 - Aula 3**
<br>
**Atividade:** Análise de vendas por região e setor — Gabarito do professor


## Objetivo prático

Aplicar os conceitos estudados na semana em uma atividade de análise de dados no PySpark.

Nesta proposta, os estudantes devem:

1. Ler um arquivo CSV com `header` e `inferSchema`.
2. Verificar o schema do DataFrame.
3. Aplicar filtros simples com `filter()`.
4. Combinar condições lógicas em um filtro.
5. Utilizar `isin()` para selecionar múltiplas regiões.
6. Agrupar dados com `groupBy()`.
7. Calcular métricas com `count()`, `sum()` e `avg()`.
8. Utilizar `agg()` para obter mais de uma métrica em uma única operação.


## Lista de materiais

- Ambiente com PySpark configurado.
- Este notebook.
- Arquivo auxiliar no mesmo diretório: `DADOSANO2C6B3S19A3_auxiliar.csv`.


## Comentário ao professor

A atividade foi organizada para retomar, em uma única prática, os principais procedimentos trabalhados nas duas aulas teóricas da semana 19.

O foco é fazer o estudante:
- ler dados e observar sua estrutura;
- aplicar filtros simples e combinados;
- reconhecer quando `isin()` é mais adequado;
- agrupar dados por uma categoria;
- interpretar métricas como quantidade, soma e média;
- perceber a utilidade de `agg()` para análises mais completas.


### Etapa 1: Ambiente pronto

A SparkSession já está criada para você focar na atividade.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum, count

spark = (
    SparkSession.builder
    .appName("S19A3_Atividade_Aluno")
    .master("local[*]")
    .getOrCreate()
)

csv_path = "DADOSANO2C6B3S19A3_auxiliar.csv"

print("SparkSession ativa")


SparkSession ativa


### Etapa 2: Leitura do arquivo

Leitura com `header=True` e `inferSchema=True`.

Comentário:
O estudante deve perceber que essas opções ajudam o Spark a interpretar corretamente os nomes das colunas e os tipos de dados.


In [2]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

df.show(8)
df.printSchema()


+--------+--------------+----------------+------------+-----------+
|id_venda|       produto|           setor|      regiao|valor_venda|
+--------+--------------+----------------+------------+-----------+
|     101|      Notebook|     Informática|     Sudeste|       3500|
|     102|         Mouse|     Informática|         Sul|         80|
|     103|       Teclado|     Informática|    Nordeste|        150|
|     104|    Smartphone|     Eletrônicos|     Sudeste|       2200|
|     105|            TV|     Eletrônicos|         Sul|       3000|
|     106|Fone de ouvido|     Eletrônicos|       Norte|        120|
|     107|     Geladeira|Eletrodomésticos|     Sudeste|       2800|
|     108|   Micro-ondas|Eletrodomésticos|Centro-Oeste|        600|
+--------+--------------+----------------+------------+-----------+
only showing top 8 rows
root
 |-- id_venda: integer (nullable = true)
 |-- produto: string (nullable = true)
 |-- setor: string (nullable = true)
 |-- regiao: string (nullable = true)


### Etapa 3: Filtro simples

Comentário:
Aqui o estudante deve identificar que `filter()` é usado para selecionar apenas as linhas que atendem a uma condição.


In [3]:
df.filter(col("valor_venda") > 1000).show()


+--------+---------------+----------------+------------+-----------+
|id_venda|        produto|           setor|      regiao|valor_venda|
+--------+---------------+----------------+------------+-----------+
|     101|       Notebook|     Informática|     Sudeste|       3500|
|     104|     Smartphone|     Eletrônicos|     Sudeste|       2200|
|     105|             TV|     Eletrônicos|         Sul|       3000|
|     107|      Geladeira|Eletrodomésticos|     Sudeste|       2800|
|     110|         Câmera|     Eletrônicos|         Sul|       1800|
|     115|Ar-condicionado|Eletrodomésticos|     Sudeste|       2500|
|     116|         Tablet|     Eletrônicos|Centro-Oeste|       1400|
+--------+---------------+----------------+------------+-----------+



### Etapa 4: Filtro com múltiplas condições

Comentário:
Nesta etapa, o estudante mobiliza o uso do operador `&` para combinar dois critérios ao mesmo tempo.


In [4]:
df.filter(
    (col("setor") == "Eletrônicos") & (col("regiao") == "Sul")
).show()


+--------+-------+-----------+------+-----------+
|id_venda|produto|      setor|regiao|valor_venda|
+--------+-------+-----------+------+-----------+
|     105|     TV|Eletrônicos|   Sul|       3000|
|     110| Câmera|Eletrônicos|   Sul|       1800|
+--------+-------+-----------+------+-----------+



### Etapa 5: Filtro com `isin()`

Comentário:
Aqui o estudante deve perceber que `isin()` simplifica filtros com mais de um valor possível na mesma coluna.


In [5]:
df.filter(
    col("regiao").isin("Sudeste", "Sul")
).show()


+--------+---------------+----------------+-------+-----------+
|id_venda|        produto|           setor| regiao|valor_venda|
+--------+---------------+----------------+-------+-----------+
|     101|       Notebook|     Informática|Sudeste|       3500|
|     102|          Mouse|     Informática|    Sul|         80|
|     104|     Smartphone|     Eletrônicos|Sudeste|       2200|
|     105|             TV|     Eletrônicos|    Sul|       3000|
|     107|      Geladeira|Eletrodomésticos|Sudeste|       2800|
|     110|         Câmera|     Eletrônicos|    Sul|       1800|
|     111|        Monitor|     Informática|Sudeste|        900|
|     114|     Ventilador|Eletrodomésticos|    Sul|        180|
|     115|Ar-condicionado|Eletrodomésticos|Sudeste|       2500|
|     117|       Roteador|     Informática|    Sul|        300|
|     118|     Smartwatch|     Eletrônicos|Sudeste|        950|
+--------+---------------+----------------+-------+-----------+



### Etapa 6: Agrupamento com `count()`

Comentário:
Nesta etapa, o estudante deve compreender que `groupBy()` organiza os grupos e `count()` calcula quantos registros existem em cada um.


In [6]:
df.groupBy("setor").count().show()


+----------------+-----+
|           setor|count|
+----------------+-----+
|     Eletrônicos|    7|
|     Informática|    6|
|Eletrodomésticos|    5|
+----------------+-----+



### Etapa 7: Agrupamentos com `sum()` e `avg()`

Comentário:
O estudante deve perceber que funções diferentes respondem a perguntas diferentes:
- `sum()` mostra o valor total;
- `avg()` mostra o valor médio.


In [7]:
df.groupBy("setor").sum("valor_venda").show()

df.groupBy("setor").avg("valor_venda").show()


+----------------+----------------+
|           setor|sum(valor_venda)|
+----------------+----------------+
|     Eletrônicos|            9720|
|     Informática|            5630|
|Eletrodomésticos|            6280|
+----------------+----------------+

+----------------+------------------+
|           setor|  avg(valor_venda)|
+----------------+------------------+
|     Eletrônicos|1388.5714285714287|
|     Informática| 938.3333333333334|
|Eletrodomésticos|            1256.0|
+----------------+------------------+



### Etapa 8: Múltiplas agregações com `agg()`

Comentário:
O objetivo é mostrar que `agg()` permite reunir várias métricas em uma única análise, o que torna o trabalho mais eficiente.


In [8]:
df.groupBy("setor").agg(
    count("valor_venda").alias("quantidade_vendas"),
    sum("valor_venda").alias("soma_valor_venda"),
    avg("valor_venda").alias("media_valor_venda")
).show()


+----------------+-----------------+----------------+------------------+
|           setor|quantidade_vendas|soma_valor_venda| media_valor_venda|
+----------------+-----------------+----------------+------------------+
|     Eletrônicos|                7|            9720|1388.5714285714287|
|     Informática|                6|            5630| 938.3333333333334|
|Eletrodomésticos|                5|            6280|            1256.0|
+----------------+-----------------+----------------+------------------+



## Sugestão de acompanhamento

Ao circular pela sala, observe se os estudantes:

- identificam corretamente a coluna usada no filtro;
- usam parênteses nas condições combinadas;
- escolhem a coluna correta para o agrupamento;
- distinguem quantidade, soma e média;
- percebem a diferença entre fazer análises separadas e usar `agg()`.
